In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
.appName("Repartition and Coalesce")\
.getOrCreate()

26/04/19 15:26:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# this data is from a python list 
data = [
(0,"Customer_0","Pune","Maharashtra","India","2023-06-29", True),
(1,"Customer_1","Bangalore","Tamil Nadu","India","2023-12-07",True),
]
 
columns = ["customer_id","name","city","state","country","registration_date","is_active"]

In [4]:
df = spark.createDataFrame(data,columns)

In [5]:
df.show()

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
+-----------+----------+---------+-----------+-------+-----------------+---------+



In [6]:
df.select('name').show()

+----------+
|      name|
+----------+
|Customer_0|
|Customer_1|
+----------+



## DataFrame Reading from HDFS

In [7]:
!hadoop fs -ls /tmp/

Found 4 items
drwxr-xr-x   - root          hadoop          0 2026-04-18 18:46 /tmp/active_cities
-rw-r--r--   2 mercy16samoei hadoop    1060750 2026-04-18 15:19 /tmp/customers.csv
drwxrwxrwt   - hdfs          hadoop          0 2026-03-17 10:41 /tmp/hadoop-yarn
drwx-wx-wx   - hive          hadoop          0 2026-03-17 10:41 /tmp/hive


In [ ]:
!hadoop fs -ls /tmp/

In [9]:
df_2 = spark.read\
.format('csv')\
.option('header', "true")\
.option('inferSchema','true')\
.load('/data/customers_100.csv')

In [10]:
df_2.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2| Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3| Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
|          5| Customer_5|Hyderabad|  Karnataka|  India|       2023-07-28|    false|
|          6| Customer_6|     Pune|      Delhi|  India|       2023-08-29|    false|
|          7| Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|     true|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-06-22|   

In [12]:
df_2.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [13]:
active_customers = df_2.filter('is_active=true')

In [14]:
active_customers.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2| Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          7| Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|     true|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-06-22|     true|
|          9| Customer_9|   Mumbai|  Telangana|  India|       2023-01-05|     true|
|         10|Customer_10|     Pune|    Gujarat|  India|       2023-08-05|     true|
|         13|Customer_13|  Chennai|  Karnataka|  India|       2023-11-06|     true|
|         15|Customer_15|   Mumbai|    Gujarat|  India|       2023-03-02|     true|
|         18|Customer_18|     Pune|      Delhi|  India|       2023-10-04|   

In [15]:
selected_columns = df_2.select('customer_id','name','city')

In [17]:
selected_columns.show()

+-----------+-----------+---------+
|customer_id|       name|     city|
+-----------+-----------+---------+
|          0| Customer_0|     Pune|
|          1| Customer_1|Bangalore|
|          2| Customer_2|Hyderabad|
|          3| Customer_3|Bangalore|
|          4| Customer_4|Ahmedabad|
|          5| Customer_5|Hyderabad|
|          6| Customer_6|     Pune|
|          7| Customer_7|Ahmedabad|
|          8| Customer_8|     Pune|
|          9| Customer_9|   Mumbai|
|         10|Customer_10|     Pune|
|         11|Customer_11|    Delhi|
|         12|Customer_12|  Chennai|
|         13|Customer_13|  Chennai|
|         14|Customer_14|Hyderabad|
|         15|Customer_15|   Mumbai|
|         16|Customer_16|  Chennai|
|         17|Customer_17|Hyderabad|
|         18|Customer_18|     Pune|
|         19|Customer_19|  Kolkata|
+-----------+-----------+---------+
only showing top 20 rows



In [18]:
spark = SparkSession.builder\
.appName("Read-Action or Transformation")\
.getOrCreate()

26/04/19 15:55:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [29]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, BooleanType, DateType, StringType

In [30]:
schema = StructType([
    StructField('customer_id',IntegerType(),False),
    StructField('name',StringType(),False),
    StructField('city',StringType(),False),
    StructField('state',StringType(),False),
    StructField('country',StringType(),False),
    StructField('registration_date',DateType(),False),
    StructField('is_active',BooleanType(),False),

])

In [22]:
df_2 = spark.read\
.format('csv')\
.option('header', "true")\
.schema(schema)\
.load('/data/customers_100.csv')

In [23]:
df.show()

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
+-----------+----------+---------+-----------+-------+-----------------+---------+



## DDL Schema 

In [24]:
ddl_schema = 'customer_id INT, name STRING, city STRING, state STRING, country STRING, regstration_date STRING, is_active BOOLEAN'

In [25]:
df_ddl = spark.read\
.format('csv')\
.option('header', "true")\
.schema(ddl_schema)\
.load('/data/customers_100.csv')

In [26]:
df_ddl.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- regstration_date: string (nullable = true)
 |-- is_active: boolean (nullable = true)



In [27]:
df_ddl.show()

+-----------+-----------+---------+-----------+-------+----------------+---------+
|customer_id|       name|     city|      state|country|regstration_date|is_active|
+-----------+-----------+---------+-----------+-------+----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|      2023-06-29|    false|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|      2023-12-07|     true|
|          2| Customer_2|Hyderabad|    Gujarat|  India|      2023-10-27|     true|
|          3| Customer_3|Bangalore|  Karnataka|  India|      2023-10-17|    false|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|      2023-03-14|    false|
|          5| Customer_5|Hyderabad|  Karnataka|  India|      2023-07-28|    false|
|          6| Customer_6|     Pune|      Delhi|  India|      2023-08-29|    false|
|          7| Customer_7|Ahmedabad|West Bengal|  India|      2023-12-28|     true|
|          8| Customer_8|     Pune|  Karnataka|  India|      2023-06-22|     true|
|   

## Read Modes In Spark

## Write Modes In Spark

In [31]:
spark = SparkSession.builder\
.appName("Write in Spark")\
.getOrCreate()

26/04/19 18:05:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [33]:
data = [(1, "Talia", "Mumbai","2025-01-12", True),
        (2, "Sandra","Delhi","2026-05-10", False),
        (2, "Hope","Dubai","2024-06-19", True)]

columns = ["customer_id","name","city","registration_date","is_active"]

In [34]:
df = spark.createDataFrame(data, columns)

In [35]:
df.write\
.format('csv')\
.option('header','true')\
.save('/data/write_output.csv')


In [37]:
!hadoop fs -ls /data/write_output.csv

Found 3 items
-rw-r--r--   2 root hadoop          0 2026-04-19 18:12 /data/write_output.csv/_SUCCESS
-rw-r--r--   2 root hadoop         81 2026-04-19 18:12 /data/write_output.csv/part-00000-cd381ddc-c8c8-413b-9991-fa2adcf1e2bf-c000.csv
-rw-r--r--   2 root hadoop        111 2026-04-19 18:12 /data/write_output.csv/part-00001-cd381ddc-c8c8-413b-9991-fa2adcf1e2bf-c000.csv


In [38]:
!hadoop fs -cat /data/write_output.csv/*

customer_id,name,city,registration_date,is_active
1,Talia,Mumbai,2025-01-12,true
customer_id,name,city,registration_date,is_active
2,Sandra,Delhi,2026-05-10,false
2,Hope,Dubai,2024-06-19,true
